# Volatility Arbitrage Strategy Backtest
This notebook runs a historical backtest for the Volatility Arbitrage (Gamma Scalping) strategy.
It uses Parkinson Volatility for RV and a proxy for IV to simulate the Variance Risk Premium.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import logging

# Add project root to path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_handler import DataHandler
from src.strategies.volatility_arb import VolatilityArbitrageStrategy
from src.options.data_handler import OptionDataHandler
from src.options.multileg import MultiLegExecutionHelper

# Setup Logging to print to stdout
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

ModuleNotFoundError: No module named 'strategies'

In [ ]:
class NoOpExecutionHandler:
    def submit_order(self, symbol, side, qty):
        print(f"[MockExec] {side.upper()} {qty} {symbol}")
    def get_all_positions(self):
        return []
    def close_position(self, symbol):
        print(f"[MockExec] Closing {symbol}")

class NoOpPortfolioManager:
    def check_allocation(self, strategy_id):
        return True

In [ ]:
# Configuration
SYMBOL = 'SPY'
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'

# Initialize Handlers
data_handler = DataHandler(paper_trading=True)
execution_handler = NoOpExecutionHandler()
portfolio_manager = NoOpPortfolioManager()
option_data_handler = OptionDataHandler(data_handler)
multi_leg_helper = MultiLegExecutionHelper(execution_handler)

# Initialize Strategy
strategy = VolatilityArbitrageStrategy(
    data_handler=data_handler,
    execution_handler=execution_handler,
    portfolio_manager=portfolio_manager,
    strategy_id=f"VolArb_{SYMBOL}",
    symbol=SYMBOL,
    option_data_handler=option_data_handler,
    multi_leg_execution=multi_leg_helper,
    lookback_days=30,
    entry_threshold=1.25
)

print(f"Strategy {strategy.strategy_id} initialized.")

# Run Backtest
print("Running backtest...")
strategy.run_backtest(START_DATE, END_DATE)